# Steam Games Data Cleaning

I’m using the full JSON dataset in this notebook. I’m still learning the cleaning process, so I’m checking the data step by step before saving a version I can use later.

The general order is: load the data, look for problems, clean the fields, check the results, and export the cleaned file.

Why: Keeping the steps in this order lets me compare the cleaned data with the original data and explain each decision instead of changing values without checking them.

## Load the full JSON dataset

The first cells load the JSON file and turn its AppID-keyed records into a DataFrame. I’m using the JSON because the CSV had shifted columns, so the values did not always line up with the headers.

Why: If the fields are shifted before cleaning starts, later calculations could look valid while actually using the wrong columns. The JSON gives me a more trustworthy starting point.

In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
json_path = Path("../Steam_Dataset/games.json")

with json_path.open("r", encoding="utf-8") as file:
    games = json.load(file)

df = (
    pd.DataFrame.from_dict(games, orient="index")
    .rename_axis("AppID")
    .reset_index()
)

## Check the raw dataset

Before changing anything, I’m checking how many rows there are and whether AppIDs are unique. This gives me a starting point to compare against after cleaning.

Why: The row count and AppID checks help me confirm that cleaning does not accidentally remove records or create duplicate games.

In [3]:
print("Shape:", df.shape)
print("Unique AppIDs:", df["AppID"].nunique())
print("Duplicate AppIDs:", df["AppID"].duplicated().sum())
print("Missing AppIDs:", df["AppID"].isna().sum())

df.head()

Shape: (141900, 43)
Unique AppIDs: 141900
Duplicate AppIDs: 0
Missing AppIDs: 0


,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


## Review raw missingness

This is my first look at missing values. A blank value is not always stored as `NaN` in this dataset; it can also be an empty string, zero, empty list, or empty dictionary. I’ll handle those cases in the next sections.

Why: Missingness affects which comparisons are fair. I’m checking the different ways missing information appears before deciding how each field should be handled.

In [4]:
full_missing_profile = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
)

full_missing_profile.head(20)

AppID                   0.0
name                    0.0
release_date            0.0
required_age            0.0
price                   0.0
dlc_count               0.0
detailed_description    0.0
about_the_game          0.0
short_description       0.0
reviews                 0.0
header_image            0.0
website                 0.0
support_url             0.0
support_email           0.0
windows                 0.0
mac                     0.0
linux                   0.0
metacritic_score        0.0
metacritic_url          0.0
achievements            0.0
Name: missing_pct, dtype: float64

## Standardize types and dates

Some fields need to be converted before I can use them for analysis. These cells convert numeric columns and turn release dates into dates. I’m also adding `release_year` because it should be easier to use for time-based questions.

Why: Consistent types make calculations possible and reduce the chance of comparing text values as if they were numbers. The new year field is a convenience for grouping games over time.

In [5]:
numeric_columns = [
    "price",
    "required_age",
    "dlc_count",
    "metacritic_score",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "achievements",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "discount",
    "peak_ccu",
]

full_zero_profile = (
    df[numeric_columns]
    .eq(0)
    .mean()
    .sort_values(ascending=False)
    .rename("zero_rate")
)

full_zero_profile

user_score                  0.999718
required_age                0.990620
average_playtime_2weeks     0.971557
median_playtime_2weeks      0.971557
metacritic_score            0.969866
peak_ccu                    0.861247
dlc_count                   0.857400
recommendations             0.846723
median_playtime_forever     0.814863
average_playtime_forever    0.814863
negative                    0.524679
achievements                0.477710
positive                    0.433608
discount                    0.274771
price                       0.201388
Name: zero_rate, dtype: float64

In [6]:
clean_df = df.copy()

clean_df["AppID"] = pd.to_numeric(
    clean_df["AppID"],
    errors="coerce"
).astype("Int64")

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

clean_df["release_date"] = pd.to_datetime(
    clean_df["release_date"],
    errors="coerce",
    format="mixed"
)

clean_df["release_year"] = (
    clean_df["release_date"].dt.year
)

In [7]:
print("Shape:", clean_df.shape)
print("Invalid dates:", clean_df["release_date"].isna().sum())
print("Missing AppIDs:", clean_df["AppID"].isna().sum())

clean_df[
    ["AppID", "name", "release_date", "release_year"]
].head()

Shape: (141900, 44)
Invalid dates: 0
Missing AppIDs: 0


,AppID,name,release_date,release_year
0,2539430,Black Dragon Mage Playtest,2023-08-01,2023
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,2016
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,2019
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,2024
4,3631080,Maze Quest VR,2025-04-24,2025


## Create review features

I’m making a few extra columns from the review information. `review_count` adds positive and negative reviews together, and `positive_ratio` shows the share of positive reviews when reviews exist. The original columns stay in the data.

Why: These features make it easier to compare review volume and review sentiment. A positive ratio based on very few reviews should be treated carefully, so I’m keeping the total review count too.

In [8]:
clean_df["review_count"] = (
    clean_df["positive"] + clean_df["negative"]
)

clean_df["positive_ratio"] = (
    clean_df["positive"]
    / clean_df["review_count"].where(
        clean_df["review_count"] > 0
    )
)

clean_df["is_free"] = clean_df["price"].eq(0)

clean_df["has_metacritic"] = (
    clean_df["metacritic_score"] > 0
)

clean_df["has_playtime"] = (
    clean_df["average_playtime_forever"] > 0
)

## Parse estimated owner ranges

The owner information is stored as text such as `0 - 20000`, so it cannot be used directly in calculations. I’m splitting it into a lower value, an upper value, and a midpoint. The midpoint is only a rough estimate, not an exact number of owners.

Why: The separate bounds preserve the uncertainty in the source while giving me numeric fields for broad comparisons. I should not describe the midpoint as sales or an exact player count.

In [9]:
owners_text = (
    clean_df["estimated_owners"]
    .astype("string")
    .str.strip()
)

owner_parts = owners_text.str.extract(
    r"^\s*([\d,]+)\s*-\s*([\d,]+)\s*$"
)

owner_parts.columns = [
    "owners_lower_text",
    "owners_upper_text",
]

invalid_owner_ranges = clean_df[
    owner_parts.isna().any(axis=1)
]

print("Invalid owner ranges:", len(invalid_owner_ranges))

Invalid owner ranges: 0


In [10]:
clean_df["owners_lower"] = pd.to_numeric(
    owner_parts["owners_lower_text"]
    .str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

clean_df["owners_upper"] = pd.to_numeric(
    owner_parts["owners_upper_text"]
    .str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

clean_df["owners_midpoint"] = (
    clean_df["owners_lower"]
    + clean_df["owners_upper"]
) / 2

In [11]:
invalid_owner_logic = clean_df[
    (clean_df["owners_lower"] > clean_df["owners_upper"])
    | clean_df["owners_lower"].isna()
    | clean_df["owners_upper"].isna()
]

print("Invalid owner logic:", len(invalid_owner_logic))

Invalid owner logic: 0


In [12]:
clean_df[
    [
        "owners_lower",
        "owners_upper",
        "owners_midpoint",
        "review_count",
        "positive_ratio",
    ]
].describe()

,owners_lower,owners_upper,owners_midpoint,review_count,positive_ratio
count,141900.0,141900.0,141900.0,1.419000e+05,82982.000000
mean,43231.501057,105217.477097,74224.489077,1.049397e+03,0.758239
std,776980.999922,1623456.608633,1199179.64858,3.034043e+04,0.238685
min,0.0,0.0,0.0,0.000000e+00,0.000000
25%,0.0,0.0,0.0,0.000000e+00,0.649613
50%,0.0,20000.0,10000.0,3.000000e+00,0.818182
75%,0.0,20000.0,10000.0,3.500000e+01,0.944444
max,100000000.0,200000000.0,150000000.0,8.815087e+06,1.000000


## Handle zero values carefully

A zero can mean different things here. For scores, CCU, and playtime, it often means that the value is not available. I’m keeping the original values, but I’m also creating flags so I can tell when a value is actually present.

Why: Replacing every zero with a missing value would be wrong for fields such as price. Keeping the original values and adding analysis versions lets me choose the appropriate interpretation later.

In [13]:
zero_as_unavailable = [
    "user_score",
    "metacritic_score",
    "peak_ccu",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
]

for column in zero_as_unavailable:
    clean_df[f"{column}_available"] = (
        clean_df[column] > 0
    )

    clean_df[f"{column}_analysis"] = (
        clean_df[column].mask(clean_df[column] == 0)
    )

In [14]:
availability_columns = [
    f"{column}_available"
    for column in zero_as_unavailable
]

availability_summary = pd.DataFrame({
    "available_count": clean_df[availability_columns].sum(),
    "available_pct": clean_df[availability_columns].mean().mul(100),
})

availability_summary

,available_count,available_pct
user_score_available,40,0.028189
metacritic_score_available,4276,3.013390
peak_ccu_available,19689,13.875264
average_playtime_forever_available,26271,18.513742
average_playtime_2weeks_available,4036,2.844257
median_playtime_forever_available,26271,18.513742
median_playtime_2weeks_available,4036,2.844257


## Clean optional text fields

Some text fields are optional and contain blank strings. I’m changing those blanks to proper missing values so they are easier to count and filter later. I’m not treating missing descriptions or links as errors.

Why: A blank description or support link can be a real limitation for a particular question, but it is not necessarily a damaged record. Converting blanks to missing values makes that limitation measurable.

In [15]:
text_columns = [
    "detailed_description",
    "about_the_game",
    "short_description",
    "reviews",
    "header_image",
    "website",
    "support_url",
    "support_email",
    "metacritic_url",
    "notes",
]

text_profile = []

for column in text_columns:
    values = clean_df[column].astype("string").str.strip()

    text_profile.append({
        "column": column,
        "empty_string_pct": values.eq("").mean() * 100,
        "literal_zero_count": values.eq("0").sum(),
        "unique_values": values.nunique(dropna=True),
    })

text_profile = (
    pd.DataFrame(text_profile)
    .set_index("column")
    .sort_values("empty_string_pct", ascending=False)
)

text_profile.round(2)

,empty_string_pct,literal_zero_count,unique_values
column,,,
metacritic_url,96.99,0,4181
reviews,91.12,0,12393
notes,80.98,0,22476
website,61.32,0,43843
support_url,57.00,0,40156
support_email,17.12,0,71773
about_the_game,6.00,0,132708
detailed_description,5.98,0,132754
short_description,5.87,0,132123


In [16]:
for column in text_columns:
    values = clean_df[column].astype("string").str.strip()
    clean_df[column] = values.mask(values.eq(""), pd.NA)

text_missing_profile = (
    clean_df[text_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
)

text_missing_profile.round(2)

metacritic_url          96.99
reviews                 91.12
notes                   80.98
website                 61.32
support_url             57.00
support_email           17.12
about_the_game           6.00
detailed_description     5.98
short_description        5.87
header_image             0.06
Name: missing_pct, dtype: float64

## Inspect and clean nested fields

Some columns contain more than one value per game. Developers, publishers, genres, and categories are lists, while tags are dictionaries when they are available. I’m checking their structure first, then changing empty lists or dictionaries to missing values.

Why: A game can belong to several genres or have several tags, so flattening these fields without checking them could create incorrect counts. Empty nested values are treated as missing because they do not provide a usable category or tag.

In [17]:
nested_columns = [
    "developers",
    "publishers",
    "genres",
    "categories",
    "tags",
]

nested_profile = pd.DataFrame({
    "non_empty_count": {
        column: clean_df[column].apply(
            lambda value: isinstance(value, (list, dict)) and len(value) > 0
        ).sum()
        for column in nested_columns
    },
    "non_empty_pct": {
        column: clean_df[column].apply(
            lambda value: isinstance(value, (list, dict)) and len(value) > 0
        ).mean() * 100
        for column in nested_columns
    },
    "avg_items_when_non_empty": {
        column: clean_df[column].apply(
            lambda value: len(value)
            if isinstance(value, (list, dict)) and len(value) > 0
            else pd.NA
        ).mean()
        for column in nested_columns
    },
}).round(2)

nested_profile

,non_empty_count,non_empty_pct,avg_items_when_non_empty
developers,133447,94.04,1.09
publishers,133048,93.76,1.04
genres,133465,94.06,2.88
categories,132923,93.67,4.77
tags,83379,58.76,14.16


In [18]:
nested_type_profile = pd.DataFrame({
    column: clean_df[column]
    .map(lambda value: type(value).__name__)
    .value_counts()
    for column in nested_columns
}).fillna(0).astype(int)

nested_type_profile

,developers,publishers,genres,categories,tags
dict,0,0,0,0,83379
list,141900,141900,141900,141900,58521


In [19]:
def empty_nested_to_na(value):
    if isinstance(value, (list, dict)) and len(value) == 0:
        return pd.NA
    return value


for column in nested_columns:
    clean_df[column] = clean_df[column].map(empty_nested_to_na)
    
nested_missing_profile = (
    clean_df[nested_columns]
    .isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .round(2)
)

nested_missing_profile

developers     5.96
publishers     6.24
genres         5.94
categories     6.33
tags          41.24
Name: missing_pct, dtype: float64

## Validate the cleaned data

At this point, I’m checking whether any values are outside reasonable ranges and whether the new columns agree with the original data. If these checks return zero invalid records, it means I did not find a problem in these checks.

Why: Validation helps catch mistakes made during cleaning. A zero result means these specific checks passed; it does not prove that every possible data-quality issue has been solved.

In [20]:
range_checks = {
    "price": clean_df["price"].ge(0),
    "required_age": clean_df["required_age"].ge(0),
    "metacritic_score": clean_df["metacritic_score"].between(0, 100),
    "user_score": clean_df["user_score"].between(0, 100),
    "positive": clean_df["positive"].ge(0),
    "negative": clean_df["negative"].ge(0),
    "recommendations": clean_df["recommendations"].ge(0),
    "achievements": clean_df["achievements"].ge(0),
    "dlc_count": clean_df["dlc_count"].ge(0),
    "discount": clean_df["discount"].between(0, 100),
    "peak_ccu": clean_df["peak_ccu"].ge(0),
    "average_playtime_forever": clean_df["average_playtime_forever"].ge(0),
    "average_playtime_2weeks": clean_df["average_playtime_2weeks"].ge(0),
    "median_playtime_forever": clean_df["median_playtime_forever"].ge(0),
    "median_playtime_2weeks": clean_df["median_playtime_2weeks"].ge(0),
    "positive_ratio": (
        clean_df["positive_ratio"].between(0, 1)
        | clean_df["positive_ratio"].isna()
    ),
}

range_check_summary = pd.Series({
    column: (~check).sum()
    for column, check in range_checks.items()
}).rename("invalid_count")

range_check_summary

price                       0
required_age                0
metacritic_score            0
user_score                  0
positive                    0
negative                    0
recommendations             0
achievements                0
dlc_count                   0
discount                    0
peak_ccu                    0
average_playtime_forever    0
average_playtime_2weeks     0
median_playtime_forever     0
median_playtime_2weeks      0
positive_ratio              0
Name: invalid_count, dtype: int64

In [21]:
final_checks = pd.Series({
    "duplicate_appids": clean_df["AppID"].duplicated().sum(),
    "missing_appids": clean_df["AppID"].isna().sum(),
    "missing_names": clean_df["name"].isna().sum(),
    "invalid_release_dates": clean_df["release_date"].isna().sum(),
    "invalid_owner_ranges": (
        clean_df["owners_lower"] > clean_df["owners_upper"]
    ).sum(),
    "invalid_owner_midpoints": (
        ~clean_df["owners_midpoint"].between(
            clean_df["owners_lower"],
            clean_df["owners_upper"]
        )
    ).sum(),
    "invalid_review_counts": (
        clean_df["review_count"]
        != clean_df["positive"] + clean_df["negative"]
    ).sum(),
    "invalid_positive_ratios": (
        clean_df["positive_ratio"].notna()
        & ~clean_df["positive_ratio"].between(0, 1)
    ).sum(),
}).rename("invalid_count")

final_checks

duplicate_appids           0
missing_appids             0
missing_names              0
invalid_release_dates      0
invalid_owner_ranges       0
invalid_owner_midpoints    0
invalid_review_counts      0
invalid_positive_ratios    0
Name: invalid_count, dtype: int64

## Prepare the final export

The `score_rank` field contains a mix of blank values and numbers. I’m converting it to a numeric column so the Parquet file can save it correctly. Values that cannot be converted will become missing.

Why: Consistent numeric types are needed for a reliable export. Turning invalid entries into missing values is safer than forcing text into a numeric value.

In [22]:
clean_df["score_rank"] = pd.to_numeric(
    clean_df["score_rank"],
    errors="coerce",
)

print(clean_df["score_rank"].dtype)
print("Available score ranks:", clean_df["score_rank"].notna().sum())

float64
Available score ranks: 40


## Preserve nested fields during export

Parquet works best with regular columns. I’m storing the nested lists and dictionaries as JSON text so their contents do not get reshaped when the file is reloaded. Notebook 3 will turn them back into Python lists and dictionaries.

Why: This preserves the original tag and multi-value contents. Without this step, a dictionary column can be expanded into a shared set of keys and make every tag appear to belong to every game.

In [23]:
def serialize_nested(value):
    if value is None or value is pd.NA:
        return None
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return None
    return json.dumps(value, ensure_ascii=False)


for column in nested_columns:
    clean_df[column] = (
        clean_df[column]
        .map(serialize_nested)
        .astype("string")
    )

## Save and verify the processed dataset

I’m saving the cleaned data as a Parquet file. After saving it, I’m loading it again and checking the shape and AppIDs. This makes sure the export did not lose rows or create duplicate IDs.

Why: The saved file is the handoff to the next notebook, so it needs to reload with the same records and key fields.

In [24]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "steam_games_clean.parquet"

clean_df.to_parquet(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {clean_df.shape}")

Saved to: ..\data\processed\steam_games_clean.parquet
Shape: (141900, 66)


In [25]:
saved_df = pd.read_parquet(output_path)

print("Saved shape:", saved_df.shape)
print("Shape matches:", saved_df.shape == clean_df.shape)
print("Unique AppIDs:", saved_df["AppID"].nunique())
print("Missing AppIDs:", saved_df["AppID"].isna().sum())
print("Duplicate AppIDs:", saved_df["AppID"].duplicated().sum())

Saved shape: (141900, 66)
Shape matches: True
Unique AppIDs: 141900
Missing AppIDs: 0
Duplicate AppIDs: 0


## Final cleaning summary

The cleaning checks passed for this dataset.

- There are 141,900 rows and the AppIDs are unique.
- The release dates and owner ranges were converted successfully.
- I added review, owner, and availability fields for later analysis.
- Blank text and empty nested values were marked as missing.
- The numeric and consistency checks returned zero invalid records.
- The cleaned file was saved to `data/processed/steam_games_clean.parquet`.

I’m ready to use this file for the exploratory analysis notebook.

The cleaned file is prepared for exploration, but the limitations found during reconnaissance still apply. The next notebook should keep owner estimates, zero values, missing fields, and multi-value columns in mind when interpreting results.